### ALL LIBRARIES

In [10]:
from __future__ import annotations

# Standard Libraries
import os
import re
import csv
import json
import random
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Optional, Union, Any
from concurrent.futures import ThreadPoolExecutor, as_completed

# Science & Geometry Libraries
import numpy as np
import pandas as pd
from scipy.spatial import KDTree
from ase import Atoms
from ase.io import read, write
from numpy.linalg import norm
from tqdm import tqdm

# Machine Learning & Evaluation
from sklearn.model_selection import (
    train_test_split, GroupShuffleSplit, StratifiedGroupKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_sample_weight
import joblib

# Gradient Boosting
import xgboost as xgb
import lightgbm as lgb

# Evaluation Metrics & Plotting
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_score, recall_score, accuracy_score, matthews_corrcoef,
    average_precision_score, RocCurveDisplay, PrecisionRecallDisplay
)
import matplotlib.pyplot as plt

# ONNX Conversion & Runtime
import onnxruntime as ort
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType as SklFloatTensorType
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType as OnnxmlFloatTensorType

# --- Global Configurations ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Ignore specific warnings for clean logs
warnings.filterwarnings("ignore", message="crystal system 'triclinic' is not interpreted*")
warnings.filterwarnings("ignore", category=UserWarning)

# Matplotlib publication-grade defaults
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "font.family": "sans-serif"
})


# One Flow

### Align along X and in XZ Plane

In [6]:
# batch_align_cif_com.py
import os
import csv
import numpy as np
import sys
from ase.io import read, write
from tqdm import tqdm
from concurrent.futures import as_completed

# ================= USER SETTINGS =================
# Set paths dynamically relative to the current directory (assumes script is in 'script' folder)
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Fallback for Jupyter environment where __file__ is undefined
    SCRIPT_DIR = os.getcwd()

BASE_DIR = os.path.abspath(os.path.join(SCRIPT_DIR, "..", "data"))
OUT_DIR  = os.path.abspath(os.path.join(SCRIPT_DIR, "..", "results", "aligned"))

folder_names = [
    "r0", "r20", "r40", "r60", "r80", "r100", "r120", "r140", "r160",
    "r180", "r200", "r220", "r240", "r260", "r280", "r300", "r320", "r340"
]

plane_mode = "nearest"   # "nearest" OR "x" / "y" / "z"
log_filename = "alignment_log.csv"
# =================================================

axis_map = {"x": 0, "y": 1, "z": 2}


# -------------------------------------------------
# Helper: Detect Jupyter environment
# -------------------------------------------------
def is_jupyter():
    try:
        shell = get_ipython().__class__.__name__
        if shell == 'ZMQInteractiveShell':
            return True
        return False
    except NameError:
        return False


# -------------------------------------------------
# Utility: Choose alignment axis
# -------------------------------------------------
def choose_axis_from_com(com, mode="nearest"):
    if mode in axis_map:
        return axis_map[mode]
    return int(np.argmin(np.abs(com)))  # nearest coordinate plane


# -------------------------------------------------
# Core Math: Custom Position Wrapping (PyTorch/NumPy)
# -------------------------------------------------
def wrap_positions_custom(positions, cell, pbc, center=(0.5, 0.5, 0.5), use_torch=False, device='cpu'):
    """
    Wraps positions back into the unit cell. Uses PyTorch GPU if available, 
    otherwise falls back to CPU NumPy.
    """
    if use_torch:
        import torch
        pos_t = torch.tensor(positions, dtype=torch.float32, device=device)
        cell_t = torch.tensor(cell, dtype=torch.float32, device=device)
        pbc_t = torch.tensor(pbc, dtype=torch.bool, device=device)
        center_t = torch.tensor(center, dtype=torch.float32, device=device)
        
        # Fractional coordinates: frac = pos_t @ inv(cell)
        inv_cell = torch.linalg.inv(cell_t)
        frac = torch.matmul(pos_t, inv_cell)
        
        # Wrap fractional coordinates along periodic boundary axes
        for i in range(3):
            if pbc_t[i]:
                shift = frac[:, i] - center_t[i] + 0.5
                frac[:, i] = (shift % 1.0) + center_t[i] - 0.5
                
        # Convert back to Cartesian coordinates
        wrapped_pos = torch.matmul(frac, cell_t)
        return wrapped_pos.cpu().numpy()
    else:
        # Fallback to NumPy
        inv_cell = np.linalg.inv(cell)
        frac = np.dot(positions, inv_cell)
        for i in range(3):
            if pbc[i]:
                shift = frac[:, i] - center[i] + 0.5
                frac[:, i] = (shift % 1.0) + center[i] - 0.5
        wrapped_pos = np.dot(frac, cell)
        return wrapped_pos


# -------------------------------------------------
# Core Alignment Function (GPU/CPU-aware)
# -------------------------------------------------
def push_com_to_plane_safe_opt(atoms, plane="nearest", use_torch=False, device='cpu'):
    """
    Safely aligns the center of mass of the atoms to the nearest coordinate plane.
    Prevents state mutation leakage by copying the Atoms object inside the function.
    """
    # Work on a copy of the structure to prevent in-place modification leakage
    atoms = atoms.copy()

    if atoms.get_cell().volume == 0:
        raise ValueError("Structure has zero cell volume.")

    original_pbc = atoms.pbc.copy()
    positions = atoms.get_positions()
    cell = atoms.get_cell()
    pbc = atoms.pbc
    masses = atoms.get_masses()

    # Wrap positions
    unwrapped = wrap_positions_custom(positions, cell, pbc, center=(0.5, 0.5, 0.5), use_torch=use_torch, device=device)
    atoms.set_positions(unwrapped)

    # Disable PBC temporarily for safe COM calculation
    atoms.pbc = False

    # Calculate center of mass
    if use_torch:
        import torch
        pos_t = torch.tensor(unwrapped, dtype=torch.float32, device=device)
        mass_t = torch.tensor(masses, dtype=torch.float32, device=device)
        com_before = (torch.matmul(mass_t, pos_t) / mass_t.sum()).cpu().numpy()
    else:
        com_before = np.dot(masses, unwrapped) / masses.sum()

    axis = choose_axis_from_com(com_before, plane)

    shift = np.zeros(3)
    shift[axis] = -com_before[axis]

    atoms.translate(shift)

    # Restore PBC
    atoms.pbc = original_pbc

    # Wrap atoms cleanly back into unit cell
    new_positions = atoms.get_positions()
    wrapped = wrap_positions_custom(new_positions, cell, original_pbc, center=(0.5, 0.5, 0.5), use_torch=use_torch, device=device)
    atoms.set_positions(wrapped)

    # Calculate final COM after translation
    if use_torch:
        import torch
        pos_t = torch.tensor(wrapped, dtype=torch.float32, device=device)
        mass_t = torch.tensor(masses, dtype=torch.float32, device=device)
        com_after = (torch.matmul(mass_t, pos_t) / mass_t.sum()).cpu().numpy()
    else:
        com_after = np.dot(masses, wrapped) / masses.sum()

    return atoms, axis, com_before, com_after, shift


# -------------------------------------------------
# Find all CIFs recursively
# -------------------------------------------------
def find_all_cifs(base_dir, rfolders):
    for rf in rfolders:
        root = os.path.join(base_dir, rf)
        if not os.path.isdir(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            for fn in filenames:
                if fn.lower().endswith(".cif"):
                    abspath = os.path.join(dirpath, fn)
                    relpath = os.path.relpath(abspath, base_dir)
                    yield abspath, relpath


# -------------------------------------------------
# Structure Validation
# -------------------------------------------------
def validate_structure(atoms):
    if len(atoms) == 0:
        raise ValueError("Structure has zero atoms.")
    if atoms.get_cell().volume <= 0:
        raise ValueError("Invalid unit cell.")
    if np.isnan(atoms.get_positions()).any():
        raise ValueError("NaN detected in atomic positions.")


# -------------------------------------------------
# Worker: Process a single structure (process/thread-safe)
# -------------------------------------------------
def process_single_file(cif_path, relpath, plane_mode, out_dir, use_torch, device):
    try:
        atoms = read(cif_path)
        validate_structure(atoms)

        atoms, axis, before, after, shift = push_com_to_plane_safe_opt(
            atoms, plane_mode, use_torch=use_torch, device=device
        )

        validate_structure(atoms)

        out_path = os.path.join(out_dir, relpath)
        os.makedirs(os.path.dirname(out_path), exist_ok=True)

        write(out_path, atoms)

        return {
            "status": "success",
            "relpath": relpath,
            "axis": axis,
            "before": list(before),
            "shift": list(shift),
            "after": list(after)
        }
    except Exception as e:
        return {
            "status": "failed",
            "cif_path": cif_path,
            "error": str(e)
        }


# -------------------------------------------------
# Main Execution
# -------------------------------------------------
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    log_path = os.path.join(OUT_DIR, log_filename)

    # 1. Detect GPU availability (PyTorch)
    try:
        import torch
        has_torch = True
        device = "cuda" if torch.cuda.is_available() else "cpu"
    except ImportError:
        has_torch = False
        device = "cpu"

    # 2. Select execution pool based on platform and notebook environment
    # Using ThreadPoolExecutor in Windows Jupyter Notebook prevents pickling bugs
    is_jupyter_win = is_jupyter() and sys.platform.startswith("win")
    if is_jupyter_win:
        from concurrent.futures import ThreadPoolExecutor as Executor
        executor_type = "ThreadPoolExecutor"
    else:
        from concurrent.futures import ProcessPoolExecutor as Executor
        executor_type = "ProcessPoolExecutor"

    all_cifs = list(find_all_cifs(BASE_DIR, folder_names))
    total = len(all_cifs)

    # Determine CPU scaling
    num_workers = max(1, os.cpu_count() - 1)

    print("========== SYSTEM CONFIGURATION ==========")
    print(f"Device:         {device.upper()} (PyTorch {'available' if has_torch else 'not installed'})")
    print(f"Executor:       {executor_type}")
    print(f"Workers:        {num_workers} parallel threads/processes")
    print(f"Found files:    {total} CIF files to process.")
    print(f"Reading from:   {BASE_DIR}")
    print(f"Writing to:     {OUT_DIR}\n")

    processed, failed = 0, 0
    errors = []

    # Process files concurrently and log results sequentially to avoid race conditions
    with open(log_path, mode="w", newline="") as logfile:
        writer = csv.writer(logfile)
        writer.writerow([
            "Relative_Path",
            "Axis",
            "COM_Before_X",
            "COM_Before_Y",
            "COM_Before_Z",
            "Shift_X",
            "Shift_Y",
            "Shift_Z",
            "COM_After_X",
            "COM_After_Y",
            "COM_After_Z"
        ])

        with Executor(max_workers=num_workers) as executor:
            futures = {
                executor.submit(
                    process_single_file,
                    cif_path,
                    relpath,
                    plane_mode,
                    OUT_DIR,
                    has_torch,
                    device
                ): (cif_path, relpath)
                for cif_path, relpath in all_cifs
            }

            with tqdm(total=total, desc="Aligning CIFs", unit="file") as pbar:
                for future in as_completed(futures):
                    res = future.result()
                    if res["status"] == "success":
                        writer.writerow([
                            res["relpath"],
                            res["axis"],
                            *res["before"],
                            *res["shift"],
                            *res["after"]
                        ])
                        processed += 1
                    else:
                        failed += 1
                        errors.append((res["cif_path"], res["error"]))
                    pbar.update(1)

    print("\n========== SUMMARY ==========")
    print(f"Total CIFs:     {total}")
    print(f"Processed OK:   {processed}")
    print(f"Failed:         {failed}")
    print(f"Log file:       {log_path}")

    if failed:
        print("\nErrors (first 20):")
        for path, msg in errors[:20]:
            print(f"- {path}: {msg}")


if __name__ == "__main__":
    main()


========== SYSTEM CONFIGURATION ==========
Device:         CPU (PyTorch not installed)
Executor:       ThreadPoolExecutor
Workers:        7 parallel threads/processes
Found files:    2916 CIF files to process.
Reading from:   d:\New folder\project\data
Writing to:     d:\New folder\project\results\aligned



Aligning CIFs: 100%|██████████| 2916/2916 [10:06<00:00,  4.81file/s]


========== SUMMARY ==========
Total CIFs:     2916
Processed OK:   2916
Failed:         0
Log file:       d:\New folder\project\results\aligned\alignment_log.csv


### Lower and Upper Splitted

In [7]:
# batch_split_upper_lower_robust.py
import sys
import os
from pathlib import Path
import numpy as np
from ase.io import read, write
from ase import Atoms
from concurrent.futures import as_completed

# ================= USER CONFIG =================
# Set paths dynamically relative to the current directory (assumes script is in 'script' folder)
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    # Fallback for Jupyter environment where __file__ is undefined
    SCRIPT_DIR = Path.cwd()

BASE_DIR = SCRIPT_DIR.parent / "results" / "aligned"
OUT_DIR  = SCRIPT_DIR.parent / "results"

FOLDER_NAMES = [
    "r0", "r20", "r40", "r60", "r80", "r100", "r120", "r140", "r160",
    "r180", "r200", "r220", "r240", "r260", "r280", "r300", "r320", "r340"
]

LOWER_DIR = OUT_DIR / "chain_splitted" / "lower"
UPPER_DIR = OUT_DIR / "chain_splitted" / "upper"

WANTED = {"C", "H", "O"}
# =================================================


# -------------------------------------------------
# Helper: Detect Jupyter environment
# -------------------------------------------------
def is_jupyter():
    try:
        shell = get_ipython().__class__.__name__
        if shell == 'ZMQInteractiveShell':
            return True
        return False
    except NameError:
        return False


# -------------------------------------------------
# Validation
# -------------------------------------------------
def validate_structure(atoms: Atoms):
    if atoms is None or len(atoms) == 0:
        raise ValueError("Empty structure.")
    if atoms.get_cell().volume <= 0:
        raise ValueError("Invalid or zero-volume cell.")
    if np.isnan(atoms.get_positions()).any():
        raise ValueError("NaN detected in positions.")


# -------------------------------------------------
# Core Math: Custom Position Wrapping (PyTorch/NumPy)
# -------------------------------------------------
def wrap_positions_custom(positions, cell, pbc, center=(0.5, 0.5, 0.5), use_torch=False, device='cpu'):
    """
    Wraps positions back into the unit cell. Uses PyTorch GPU if available, 
    otherwise falls back to CPU NumPy.
    """
    if use_torch:
        import torch
        pos_t = torch.tensor(positions, dtype=torch.float32, device=device)
        cell_t = torch.tensor(cell, dtype=torch.float32, device=device)
        pbc_t = torch.tensor(pbc, dtype=torch.bool, device=device)
        center_t = torch.tensor(center, dtype=torch.float32, device=device)
        
        inv_cell = torch.linalg.inv(cell_t)
        frac = torch.matmul(pos_t, inv_cell)
        
        for i in range(3):
            if pbc_t[i]:
                shift = frac[:, i] - center_t[i] + 0.5
                frac[:, i] = (shift % 1.0) + center_t[i] - 0.5
                
        wrapped_pos = torch.matmul(frac, cell_t)
        return wrapped_pos.cpu().numpy()
    else:
        inv_cell = np.linalg.inv(cell)
        frac = np.dot(positions, inv_cell)
        for i in range(3):
            if pbc[i]:
                shift = frac[:, i] - center[i] + 0.5
                frac[:, i] = (shift % 1.0) + center[i] - 0.5
        wrapped_pos = np.dot(frac, cell)
        return wrapped_pos


# -------------------------------------------------
# Safe unwrap for Z sorting (prevents global state leak)
# -------------------------------------------------
def unwrap_atoms_for_sorting(atoms: Atoms, use_torch=False, device='cpu'):
    # Work on a copy of the structure to prevent in-place modification leakage
    atoms = atoms.copy()
    positions = atoms.get_positions()
    cell = atoms.get_cell()
    pbc = atoms.get_pbc()

    unwrapped = wrap_positions_custom(
        positions,
        cell,
        pbc,
        center=(0.5, 0.5, 0.5),
        use_torch=use_torch,
        device=device
    )
    atoms.set_positions(unwrapped)
    return atoms


# -------------------------------------------------
# Subset while preserving cell & PBC
# -------------------------------------------------
def subset_with_cell(parent_atoms: Atoms, keep_idx):
    if keep_idx is None or len(keep_idx) == 0:
        return None

    sub = parent_atoms[keep_idx]

    # preserve lattice & periodicity
    sub.set_cell(parent_atoms.cell)
    sub.set_pbc(parent_atoms.get_pbc())

    # wrap cleanly inside unit cell
    sub.wrap()

    return sub


# -------------------------------------------------
# Z-half splitting (robust)
# -------------------------------------------------
def split_by_sorted_z_half(atoms: Atoms, wanted=WANTED, use_torch=False, device='cpu'):
    validate_structure(atoms)

    atoms = unwrap_atoms_for_sorting(atoms, use_torch=use_torch, device=device)

    symbols = atoms.get_chemical_symbols()
    pos = atoms.get_positions()

    selected = [
        (i, s, p[2])
        for i, (s, p) in enumerate(zip(symbols, pos))
        if s.upper() in wanted
    ]

    if len(selected) == 0:
        selected = [(i, s, p[2]) for i, (s, p) in enumerate(zip(symbols, pos))]

    selected.sort(key=lambda t: t[2])

    N = len(selected)

    if N <= 1:
        lower_idx = np.array([i for (i, _, _) in selected], dtype=int)
        upper_idx = np.array([], dtype=int)
        return lower_idx, upper_idx

    half = N // 2
    lower_idx = np.array([i for (i, _, _) in selected[:half]], dtype=int)
    upper_idx = np.array([i for (i, _, _) in selected[half:]], dtype=int)

    return lower_idx, upper_idx


# -------------------------------------------------
# Process one file (worker function)
# -------------------------------------------------
def process_one(infile: Path, out_prefix: str, lower_dir: Path, upper_dir: Path, wanted, use_torch, device):
    if not infile.exists():
        return {"status": "skip", "infile": str(infile)}

    try:
        atoms = read(str(infile))
        validate_structure(atoms)
    except Exception as e:
        return {"status": "error", "infile": str(infile), "error": f"Failed to read: {e}"}

    try:
        lower_idx, upper_idx = split_by_sorted_z_half(atoms, wanted=wanted, use_torch=use_torch, device=device)

        lower = subset_with_cell(atoms, lower_idx)
        upper = subset_with_cell(atoms, upper_idx)

        lower_path = lower_dir / f"{out_prefix}_lower.cif"
        upper_path = upper_dir / f"{out_prefix}_upper.cif"

        if lower is not None and len(lower) > 0:
            write(str(lower_path), lower)

        if upper is not None and len(upper) > 0:
            write(str(upper_path), upper)

        lower_name = lower_path.name if (lower is not None and len(lower) > 0) else "(empty lower)"
        upper_name = upper_path.name if (upper is not None and len(upper) > 0) else "(empty upper)"
        return {
            "status": "ok",
            "infile_name": infile.name,
            "lower_name": lower_name,
            "upper_name": upper_name
        }

    except Exception as e:
        return {"status": "error", "infile": str(infile), "error": f"Failed during split: {e}"}


# -------------------------------------------------
# Main
# -------------------------------------------------
def main():
    LOWER_DIR.mkdir(parents=True, exist_ok=True)
    UPPER_DIR.mkdir(parents=True, exist_ok=True)

    # 1. Detect GPU availability (PyTorch)
    try:
        import torch
        has_torch = True
        device = "cuda" if torch.cuda.is_available() else "cpu"
    except ImportError:
        has_torch = False
        device = "cpu"

    # 2. Select execution pool based on platform and notebook environment
    # Using ThreadPoolExecutor in Windows Jupyter Notebook prevents pickling bugs
    is_jupyter_win = is_jupyter() and sys.platform.startswith("win")
    if is_jupyter_win:
        from concurrent.futures import ThreadPoolExecutor as Executor
        executor_type = "ThreadPoolExecutor"
    else:
        from concurrent.futures import ProcessPoolExecutor as Executor
        executor_type = "ProcessPoolExecutor"

    num_workers = max(1, os.cpu_count() - 1)

    print("========== SYSTEM CONFIGURATION ==========")
    print(f"Device:         {device.upper()} (PyTorch {'available' if has_torch else 'not installed'})")
    print(f"Executor:       {executor_type}")
    print(f"Workers:        {num_workers} parallel threads/processes\n")

    tasks = []
    for r in FOLDER_NAMES:
        infile = BASE_DIR / r / "t0" / "t0_0.cif"
        out_prefix = f"{r}_t0_t0_0"
        tasks.append((infile, out_prefix))

    total = len(tasks)
    print(f"Submitting {total} splitting tasks...\n")

    with Executor(max_workers=num_workers) as executor:
        futures = {
            executor.submit(
                process_one,
                infile,
                out_prefix,
                LOWER_DIR,
                UPPER_DIR,
                WANTED,
                has_torch,
                device
            ): (infile, out_prefix)
            for infile, out_prefix in tasks
        }

        for future in as_completed(futures):
            res = future.result()
            if res["status"] == "ok":
                print(f"[ok] {res['infile_name']} -> {res['lower_name']} | {res['upper_name']}")
            elif res["status"] == "skip":
                print(f"[skip] Missing file: {res['infile']}")
            elif res["status"] == "error":
                print(f"[error] {res['infile']}: {res['error']}")


if __name__ == "__main__":
    main()


========== SYSTEM CONFIGURATION ==========
Device:         CPU (PyTorch not installed)
Executor:       ThreadPoolExecutor
Workers:        7 parallel threads/processes

Submitting 18 splitting tasks...

[ok] t0_0.cif -> r0_t0_t0_0_lower.cif | r0_t0_t0_0_upper.cif
[ok] t0_0.cif -> r40_t0_t0_0_lower.cif | r40_t0_t0_0_upper.cif
[ok] t0_0.cif -> r60_t0_t0_0_lower.cif | r60_t0_t0_0_upper.cif
[ok] t0_0.cif -> r20_t0_t0_0_lower.cif | r20_t0_t0_0_upper.cif
[ok] t0_0.cif -> r80_t0_t0_0_lower.cif | r80_t0_t0_0_upper.cif
[ok] t0_0.cif -> r120_t0_t0_0_lower.cif | r120_t0_t0_0_upper.cif
[ok] t0_0.cif -> r100_t0_t0_0_lower.cif | r100_t0_t0_0_upper.cif
[ok] t0_0.cif -> r140_t0_t0_0_lower.cif | r140_t0_t0_0_upper.cif
[ok] t0_0.cif -> r220_t0_t0_0_lower.cif | r220_t0_t0_0_upper.cif
[ok] t0_0.cif -> r200_t0_t0_0_lower.cif | r200_t0_t0_0_upper.cif
[ok] t0_0.cif -> r180_t0_t0_0_lower.cif | r180_t0_t0_0_upper.cif
[ok] t0_0.cif -> r160_t0_t0_0_lower.cif | r160_t0_t0_0_upper.cif
[ok] t0_0.cif -> r260_t0_t0_0_

### Features from original data

In [8]:
# robust_cif_feature_extraction.py
import os
import re
import sys
import numpy as np
import pandas as pd
from tqdm import tqdm
from ase.io import read
from scipy.spatial import cKDTree as KDTree

# ------------------ USER PATHS ------------------
# Set paths dynamically relative to the current directory (assumes script is in 'script' folder)
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Fallback for Jupyter environment where __file__ is undefined
    SCRIPT_DIR = os.getcwd()

BASE_DIR   = os.path.abspath(os.path.join(SCRIPT_DIR, "..", "data"))
OUTPUT_DIR = os.path.abspath(os.path.join(SCRIPT_DIR, "..", "results"))
OUTPUT_CSV = "cif_bounds_summary_robust_features.csv"

# ------------------ PARAMETERS ------------------
rvw_H   = 1.20
rvw_O   = 1.53
rvw_err = 0.10


# -------------------------------------------------
# Helper: Detect Jupyter environment
# -------------------------------------------------
def is_jupyter():
    try:
        shell = get_ipython().__class__.__name__
        if shell == 'ZMQInteractiveShell':
            return True
        return False
    except NameError:
        return False


# ------------------------------------------------
# ---------------- VALIDATION --------------------
def validate_structure(atoms):
    if atoms is None or len(atoms) == 0:
        raise ValueError("Empty structure.")
    if atoms.get_cell().volume <= 0:
        raise ValueError("Invalid or zero-volume cell.")
    if np.isnan(atoms.get_positions()).any():
        raise ValueError("NaN detected in atomic positions.")


# -------------------------------------------------
# Core Math: Custom Position Wrapping (PyTorch/NumPy)
# -------------------------------------------------
def wrap_positions_custom(positions, cell, pbc, center=(0.5, 0.5, 0.5), use_torch=False, device='cpu'):
    """
    Wraps positions back into the unit cell. Uses PyTorch GPU if available, 
    otherwise falls back to CPU NumPy.
    """
    if use_torch:
        import torch
        pos_t = torch.tensor(positions, dtype=torch.float32, device=device)
        cell_t = torch.tensor(cell, dtype=torch.float32, device=device)
        pbc_t = torch.tensor(pbc, dtype=torch.bool, device=device)
        center_t = torch.tensor(center, dtype=torch.float32, device=device)
        
        inv_cell = torch.linalg.inv(cell_t)
        frac = torch.matmul(pos_t, inv_cell)
        
        for i in range(3):
            if pbc_t[i]:
                shift = frac[:, i] - center_t[i] + 0.5
                frac[:, i] = (shift % 1.0) + center_t[i] - 0.5
                
        wrapped_pos = torch.matmul(frac, cell_t)
        return wrapped_pos.cpu().numpy()
    else:
        inv_cell = np.linalg.inv(cell)
        frac = np.dot(positions, inv_cell)
        for i in range(3):
            if pbc[i]:
                shift = frac[:, i] - center[i] + 0.5
                frac[:, i] = (shift % 1.0) + center[i] - 0.5
        wrapped_pos = np.dot(frac, cell)
        return wrapped_pos


# ------------------------------------------------
# ----------- SAFE LOAD + UNWRAP -----------------
def load_atoms_safely(cif_path, use_torch=False, device='cpu'):
    atoms = read(cif_path)
    validate_structure(atoms)

    # unwrap consistently to avoid boundary splits
    positions = atoms.get_positions()
    cell = atoms.get_cell()
    pbc = atoms.get_pbc()

    unwrapped = wrap_positions_custom(
        positions,
        cell,
        pbc,
        center=(0.5, 0.5, 0.5),
        use_torch=use_torch,
        device=device
    )
    atoms.set_positions(unwrapped)
    atoms.pbc = False  # turn off periodicity for KDTree stage

    return atoms


# ------------------------------------------------
# ---------------- SORT HELPERS ------------------
def dir_key(name: str):
    m = re.search(r'[-+]?\d*\.?\d+', name)
    return (float(m.group()) if m else float('inf'), name)


def file_key(fname: str):
    m = re.search(r'(\\d+)(?=\\.cif$)', fname.lower())
    return int(m.group(1)) if m else float('inf')


# ------------------------------------------------
# --------- SPLIT BY Z (SAFER VERSION) ----------
def split_chain_by_z(df: pd.DataFrame, element: str):
    sub = df[df["Element"] == element].copy()
    if sub.empty:
        return sub.copy(), sub.copy()

    sub = sub.sort_values("z").reset_index(drop=True)

    half = len(sub) // 2
    lower = sub.iloc[:half].copy()
    upper = sub.iloc[half:].copy()

    return upper, lower


# ------------------------------------------------
# --------- PERIODIC-AWARE DISTANCES ------------
def periodic_distances(lower_coords, upper_coords, cutoff, use_torch=False, device='cpu'):
    if len(lower_coords) == 0 or len(upper_coords) == 0:
        return []

    if use_torch:
        import torch
        l_tensor = torch.tensor(lower_coords, dtype=torch.float32, device=device)
        u_tensor = torch.tensor(upper_coords, dtype=torch.float32, device=device)
        # Compute pairwise distances on GPU (size: M x N)
        dists = torch.cdist(u_tensor, l_tensor)
        min_dists, _ = torch.min(dists, dim=1)
        filtered = min_dists[min_dists <= cutoff]
        return filtered.cpu().tolist()
    else:
        tree = KDTree(lower_coords)
        dists, _ = tree.query(upper_coords)
        return [float(d) for d in dists if d <= cutoff]


# ------------------------------------------------
# ------------- MAIN FEATURE LOGIC --------------
def compute_bounds_for_cif(cif_path, rvw_H, rvw_O, rvw_err, use_torch, device):
    try:
        atoms = load_atoms_safely(cif_path, use_torch=use_torch, device=device)

        symbols = atoms.get_chemical_symbols()
        positions = atoms.get_positions()

        df = pd.DataFrame(positions, columns=["x", "y", "z"])
        df.insert(0, "Element", symbols)

        # split H and O
        uH, lH = split_chain_by_z(df, "H")
        uO, lO = split_chain_by_z(df, "O")

        uH_c = uH[["x","y","z"]].to_numpy()
        lH_c = lH[["x","y","z"]].to_numpy()
        uO_c = uO[["x","y","z"]].to_numpy()
        lO_c = lO[["x","y","z"]].to_numpy()

        cut_HH = 2 * rvw_H + rvw_err
        cut_OO = 2 * rvw_O + rvw_err
        cut_OH = rvw_O + rvw_H + rvw_err

        dist_HH = periodic_distances(lH_c, uH_c, cut_HH, use_torch=use_torch, device=device)
        dist_OO = periodic_distances(lO_c, uO_c, cut_OO, use_torch=use_torch, device=device)
        dist_OH = periodic_distances(lH_c, uO_c, cut_OH, use_torch=use_torch, device=device)
        dist_HO = periodic_distances(lO_c, uH_c, cut_OH, use_torch=use_torch, device=device)

        def mean_or_default(x, default=3.0):
            return float(np.mean(x)) if len(x) else float(default)

        return {
            "file_path": cif_path,
            "avg_HH_dist": mean_or_default(dist_HH),
            "avg_OO_dist": mean_or_default(dist_OO),
            "avg_OH_dist": mean_or_default(dist_OH),
            "avg_HO_dist": mean_or_default(dist_HO),
            "count_HH": len(dist_HH),
            "count_OO": len(dist_OO),
            "count_OH": len(dist_OH),
            "count_HO": len(dist_HO),
            "sum_of_count": len(dist_HH) + len(dist_OO) + len(dist_OH) + len(dist_HO)
        }

    except Exception as e:
        return {"file_path": cif_path, "error": str(e)}


# ------------------------------------------------
# ------------- WALK ALL FILES -------------------
def collect_all_results(base_dir, rvw_H, rvw_O, rvw_err, use_torch, device):
    # Select execution pool based on platform and notebook environment
    is_jupyter_win = is_jupyter() and sys.platform.startswith("win")
    if is_jupyter_win:
        from concurrent.futures import ThreadPoolExecutor as Executor
        executor_type = "ThreadPoolExecutor"
    else:
        from concurrent.futures import ProcessPoolExecutor as Executor
        executor_type = "ProcessPoolExecutor"

    num_workers = max(1, os.cpu_count() - 1)

    print("========== SYSTEM CONFIGURATION ==========")
    print(f"Device:         {device.upper()} (PyTorch {'available' if use_torch else 'not installed'})")
    print(f"Executor:       {executor_type}")
    print(f"Workers:        {num_workers} parallel threads/processes")
    print(f"Reading from:   {base_dir}\n")

    # Collect sorted list of file paths to process
    file_paths = []
    for root, dirs, files in os.walk(base_dir):
        dirs.sort(key=dir_key)
        for file in sorted(files, key=file_key):
            if file.lower().endswith(".cif"):
                file_paths.append(os.path.join(root, file))

    total_files = len(file_paths)
    results = []

    # Process files concurrently, but retrieve them in the exact original sorted order
    with Executor(max_workers=num_workers) as executor:
        futures = [
            executor.submit(
                compute_bounds_for_cif,
                path,
                rvw_H,
                rvw_O,
                rvw_err,
                use_torch,
                device
            )
            for path in file_paths
        ]

        with tqdm(total=total_files, desc="Processing CIF files", unit="file") as pbar:
            for future in futures:
                results.append(future.result())
                pbar.update(1)

    return pd.DataFrame(results)


# ------------------------------------------------
# -------------------- MAIN ----------------------
if __name__ == "__main__":
    # Detect GPU availability (PyTorch)
    try:
        import torch
        has_torch = True
        device = "cuda" if torch.cuda.is_available() else "cpu"
    except ImportError:
        has_torch = False
        device = "cpu"

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = collect_all_results(BASE_DIR, rvw_H, rvw_O, rvw_err, has_torch, device)

    out_path = os.path.join(OUTPUT_DIR, OUTPUT_CSV)
    df.to_csv(out_path, index=False)

    print(f"\nSaved to: {out_path}")
    print(df.head())


========== SYSTEM CONFIGURATION ==========
Device:         CPU (PyTorch not installed)
Executor:       ThreadPoolExecutor
Workers:        7 parallel threads/processes
Reading from:   d:\New folder\project\data



Processing CIF files: 100%|██████████| 2916/2916 [07:04<00:00,  6.88file/s]


Saved to: d:\New folder\project\results\cif_bounds_summary_robust_features.csv
                                     file_path  avg_HH_dist  avg_OO_dist  \
0    d:\New folder\project\data\r0\t0\t0_0.cif     2.478338      3.00000   
1  d:\New folder\project\data\r0\t0\t0_100.cif     2.434359      3.12617   
2  d:\New folder\project\data\r0\t0\t0_120.cif     2.274089      3.00000   
3  d:\New folder\project\data\r0\t0\t0_140.cif     2.002765      3.00000   
4  d:\New folder\project\data\r0\t0\t0_160.cif     2.141875      3.00000   

   avg_OH_dist  avg_HO_dist  count_HH  count_OO  count_OH  count_HO  \
0     2.469072     2.179423         2         0         2         2   
1     3.000000     2.447893         2         1         0         2   
2     3.000000     2.424823         4         0         0         2   
3     3.000000     3.000000         2         0         0         0   
4     3.000000     3.000000         2         0         0         0   

   sum_of_count  
0             6  


### Data Preprocessing for Model Input

In [ ]:
# ================= PATH CONFIGURATION =================
# Set paths dynamically relative to the current directory (assumes notebook is in 'script' folder)
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Fallback for Jupyter environment where __file__ is undefined
    SCRIPT_DIR = os.getcwd()

# Dynamic path resolution
ENERGY_PATH = os.path.abspath(os.path.join(SCRIPT_DIR, "..", "data", "file_energy.csv"))
OUTPUT_CSV  = os.path.abspath(os.path.join(SCRIPT_DIR, "..", "results", "features_original_model_ready.csv"))

# Ensure output directory exists
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)


# ================= DATA LOADING =================
# Load the features DataFrame from memory if available, otherwise read it from disk
if 'df' in globals():
    data = df.copy()
elif 'data' in globals():
    data = data.copy()
else:
    # Fallback: load features from the CSV generated in the previous step
    features_csv = os.path.join(SCRIPT_DIR, "..", "results", "cif_bounds_summary_robust_features.csv")
    print(f"Loading features from: {features_csv}")
    data = pd.read_csv(features_csv)


# ================= FILE ID CREATION =================
def make_file_id(p):
    if pd.isna(p):
        return pd.NA
    # Grab base folder, subfolder, and filename
    fname = os.path.basename(p)
    sub   = os.path.basename(os.path.dirname(p))
    base  = os.path.basename(os.path.dirname(os.path.dirname(p)))
    return f"{base}_{sub}_{fname}"

# Create new column
data["file_id"] = data["file_path"].map(make_file_id)


# ================= ENERGY DATA CLEANING =================
print(f"Reading energy data from: {ENERGY_PATH}")
energy_data = pd.read_csv(ENERGY_PATH, index_col=0)

# Replace 'transformed' strings inside file_id using regex
pattern = r'[_-]transformed(?=\.[^.]+$)'
before = energy_data['file_id'].copy()
energy_data['file_id'] = energy_data['file_id'].astype(str).str.replace(pattern, '', regex=True)

print("Changed rows in energy_data:", (before != energy_data['file_id']).sum())

# Columns to bring over from energy_data
cols = ['r_bottom', 'displacement', 'r_upper', 'adjusted_energy']


# ================= INTEGRITY CHECKS =================
assert 'file_id' in data.columns and 'file_id' in energy_data.columns, "file_id column missing!"
assert not data['file_id'].duplicated().any(), "Duplicate file_id detected in features data!"
assert not energy_data['file_id'].duplicated().any(), "Duplicate file_id detected in energy data!"


# ================= LEFT JOIN =================
# Make energy_data a lookup by file_id
energy_lookup = energy_data[['file_id'] + cols].set_index('file_id')

# Left-join while preserving data row order
data = data.join(energy_lookup, on='file_id')

# Quick integrity check
missing = data[cols].isna().any(axis=1).sum()
print(f"Joined. Rows with any missing joined values: {missing} / {len(data)}")


# ================= SAVE CLEANED DATA =================
# Drop Unnamed: 0 from data permanently if it exists
data = data.drop(columns=['Unnamed: 0'], errors='ignore')

# Save data to CSV without any index or extra columns
data.to_csv(OUTPUT_CSV, index=False)
print(f"Preprocessed dataset saved successfully to: {OUTPUT_CSV}")


Reading energy data from: d:\New folder\project\data\file_energy.csv
Changed rows in energy_data: 2916
Joined. Rows with any missing joined values: 0 / 2916
Preprocessed dataset saved successfully to: d:\New folder\project\results\features_original_model_ready.csv


### Model

In [37]:
import os
import re
import json
import random
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

# Machine Learning & Pipelines
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
import joblib

# Attempt robust import of FrozenEstimator for newer scikit-learn versions
try:
    from sklearn.calibration import FrozenEstimator
    HAS_FROZEN = True
except ImportError:
    try:
        from sklearn.utils.metaestimators import FrozenEstimator
        HAS_FROZEN = True
    except ImportError:
        HAS_FROZEN = False

# Gradient Boosting
import xgboost as xgb
import lightgbm as lgb

# Evaluation Metrics
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, matthews_corrcoef, average_precision_score,
    brier_score_loss
)

# ONNX Conversion & Runtime
import onnxruntime as ort
from skl2onnx import convert_sklearn, update_registered_converter
from skl2onnx.common.data_types import FloatTensorType
from skl2onnx.common.shape_calculator import calculate_linear_classifier_output_shapes
from onnxmltools.convert.xgboost.operator_converters.XGBoost import convert_xgboost
from onnxmltools.convert.lightgbm.operator_converters.LightGbm import convert_lightgbm

# Ignore warnings for cleaner output
warnings.filterwarnings("ignore")

# ============================================================
# 0) REPRODUCIBILITY & SETUP
# ============================================================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Setup path resolution
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()

# Find project directory (handles running from 'script' subdirectory)
PROJECT_DIR = SCRIPT_DIR if (SCRIPT_DIR / "results").exists() else SCRIPT_DIR.parent
DATA_PATH = PROJECT_DIR / "results" / "features_original_model_ready.csv"
ONNX_PATH = PROJECT_DIR / "results" / "pipeline_model_calibrated.onnx"
JOBLIB_PATH = PROJECT_DIR / "results" / "pipeline_model_calibrated.joblib"
THRESHOLD_PATH = PROJECT_DIR / "results" / "model_threshold.json"

print(f"Reading dataset from: {DATA_PATH}")
data = pd.read_csv(DATA_PATH)

# ============================================================
# 1) PREPROCESSING & FEATURE SELECTION
# ============================================================
# Target variable: adjusted_energy > 0 (1 = unstable/steric repulsion, 0 = stable)
data["label"] = (data["adjusted_energy"] > 0).astype(int)

# Drop identifier columns and sum_of_count (to address multicollinearity)
cols_to_drop = [
    "file_path", "file_id", "r_bottom", "displacement", 
    "r_upper", "adjusted_energy", "sum_of_count", "label", "group"
]
feature_cols = [c for c in data.columns if c not in cols_to_drop]

X = data[feature_cols].copy()
y = data["label"].astype(int).copy()

print("Data shape:", X.shape)
print("Features:", feature_cols)
print("Class distribution:\n", y.value_counts())

# ============================================================
# 2) TRAIN / VALIDATION / TEST SPLIT (60% Train, 20% Val, 20% Test)
# ============================================================
# Split 80% train-val / 20% test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

# Split 80% train-val into 60% train and 20% val (25% of 80% is 20%)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=SEED
)

print(f"\nSplits Summary:")
print(f"Train Shape      : {X_train.shape}")
print(f"Validation Shape : {X_val.shape}")
print(f"Test Shape       : {X_test.shape}")

# ============================================================
# 3) DEFINE HIGHLY REGULARIZED BASE ESTIMATORS
# ============================================================
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1
)

xgb_model = xgb.XGBClassifier(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2.0,
    scale_pos_weight=pos_weight,
    eval_metric="logloss",
    random_state=SEED,
    n_jobs=-1
)

lgb_model = lgb.LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=4,
    num_leaves=15,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2.0,
    class_weight="balanced",
    random_state=SEED,
    verbosity=-1,
    n_jobs=-1
)

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    alpha=1e-3,
    learning_rate_init=0.001,
    max_iter=500,
    random_state=SEED
)

# ============================================================
# 4) FIT SCALER & STACKING CLASSIFIER ON TRAIN SUBSET
# ============================================================
print("\nFitting Stacking Classifier on Training subset...")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

stack = StackingClassifier(
    estimators=[
        ("rf", rf),
        ("xgb", xgb_model),
        ("lgb", lgb_model),
        ("mlp", mlp)
    ],
    final_estimator=LogisticRegression(
        C=0.1, 
        class_weight="balanced", 
        max_iter=3000, 
        random_state=SEED
    ),
    cv=10,
    n_jobs=-1
)

stack.fit(X_train_scaled, y_train)

# ============================================================
# 5) PROBABILITY CALIBRATION ON SCALED VALIDATION DATA
# ============================================================
print("\nCalibrating Stacking Classifier on Validation subset...")
if HAS_FROZEN:
    print("Using FrozenEstimator wrapper for calibration (scikit-learn 1.6+ compatibility)")
    calibrated_stack = CalibratedClassifierCV(
        estimator=FrozenEstimator(stack),
        method="sigmoid"
    )
else:
    print("Using cv='prefit' for calibration (older scikit-learn compatibility)")
    calibrated_stack = CalibratedClassifierCV(
        estimator=stack,
        method="sigmoid",
        cv="prefit"
    )
calibrated_stack.fit(X_val_scaled, y_val)

# ============================================================
# 6) UNWRAP FROZEN ESTIMATOR REPRESENTATION FOR ONNX
# ============================================================
if HAS_FROZEN:
    for clf in calibrated_stack.calibrated_classifiers_:
        if hasattr(clf, "estimator") and hasattr(clf.estimator, "estimator"):
            clf.estimator = clf.estimator.estimator

# ============================================================
# 7) CONSTRUCT TOP-LEVEL END-TO-END PIPELINE
# ============================================================
final_pipeline = Pipeline([
    ("scaler", scaler),
    ("calibrated_stack", calibrated_stack)
])

# ============================================================
# 8) THRESHOLD TUNING ON VALIDATION (MCC OPTIMIZATION)
# ============================================================
val_probs = final_pipeline.predict_proba(X_val)[:, 1]
test_probs = final_pipeline.predict_proba(X_test)[:, 1]

thresholds = np.linspace(0.01, 0.99, 1000)
best_mcc = -1
best_threshold = 0.5

for t in thresholds:
    preds = (val_probs >= t).astype(int)
    if len(np.unique(preds)) < 2:
        continue
    mcc = matthews_corrcoef(y_val, preds)
    if mcc > best_mcc:
        best_mcc = mcc
        best_threshold = t

print("\nBest Calibrated Validation Threshold:", round(best_threshold, 4))
print("Best Calibrated Validation MCC      :", round(best_mcc, 4))

# ============================================================
# 9) EXPECTED CALIBRATION ERROR (ECE) METRIC
# ============================================================
def expected_calibration_error(y_true, probs, n_bins=10):
    y_true = np.asarray(y_true)
    probs = np.asarray(probs)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (probs >= lo) & (probs < hi)
        if i == n_bins - 1:
            mask = (probs >= lo) & (probs <= hi)
        if mask.sum() == 0:
            continue
        bin_conf = probs[mask].mean()
        bin_acc = y_true[mask].mean()
        bin_weight = mask.mean()
        ece += bin_weight * abs(bin_acc - bin_conf)
    return ece

# ============================================================
# 10) MODEL EVALUATION
# ============================================================
def evaluate_model(name, y_true, probs, threshold):
    preds = (probs >= threshold).astype(int)
    print(f"\n================ {name} ================")
    print("Decision Threshold :", round(threshold, 4))
    print("MCC                :", round(matthews_corrcoef(y_true, preds), 4))
    print("Accuracy           :", round(accuracy_score(y_true, preds), 4))
    print("ROC-AUC            :", round(roc_auc_score(y_true, probs), 4))
    print("PR-AUC (Avg Prec)  :", round(average_precision_score(y_true, probs), 4))
    print("Brier Score Loss   :", round(brier_score_loss(y_true, probs), 6))
    print("ECE                :", round(expected_calibration_error(y_true, probs), 6))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, preds))
    print("\nClassification Report:")
    print(classification_report(y_true, preds, digits=4))
    return preds

val_preds = evaluate_model("CALIBRATED VALIDATION RESULTS", y_val, val_probs, best_threshold)
test_preds = evaluate_model("CALIBRATED FINAL TEST RESULTS", y_test, test_probs, best_threshold)

# ============================================================
# 11) BOOTSTRAP MCC CONFIDENCE INTERVAL ON TEST SET
# ============================================================
def bootstrap_mcc_ci(y_true, probs, threshold, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    probs = np.asarray(probs)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        y_sample = y_true[idx]
        p_sample = probs[idx]
        pred_sample = (p_sample >= threshold).astype(int)
        if len(np.unique(y_sample)) < 2 or len(np.unique(pred_sample)) < 2:
            continue
        scores.append(matthews_corrcoef(y_sample, pred_sample))
    scores = np.array(scores)
    return float(np.mean(scores)), float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))

mean_mcc, ci_low, ci_high = bootstrap_mcc_ci(y_test, test_probs, best_threshold)
print("\n================ TEST MCC BOOTSTRAP CI ================")
print(f"Mean MCC: {mean_mcc:.4f}")
print(f"95% CI  : [{ci_low:.4f}, {ci_high:.4f}]")

# ============================================================
# 12) SAVE SKLEARN MODEL & METADATA
# ============================================================
joblib.dump(final_pipeline, JOBLIB_PATH)
metadata = {
    "threshold": float(best_threshold),
    "features": feature_cols,
    "seed": SEED,
    "model_type": "stacking_ensemble_calibrated",
    "calibration": "sigmoid_platt_scaling",
    "test_mcc": float(matthews_corrcoef(y_test, test_preds)),
    "test_mcc_bootstrap_mean": mean_mcc,
    "test_mcc_ci_low": ci_low,
    "test_mcc_ci_high": ci_high
}
with open(THRESHOLD_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print("\nSaved calibrated sklearn model to:", JOBLIB_PATH)
print("Saved threshold metadata to:", THRESHOLD_PATH)

# ============================================================
# 13) ONNX EXPORT AND VERIFICATION
# ============================================================
print("\n================ EXPORTING TO ONNX ================")

# Register XGBoost and LightGBM converters for skl2onnx
update_registered_converter(
    xgb.XGBClassifier, 'XGBClassifier',
    calculate_linear_classifier_output_shapes, convert_xgboost,
    options={'nocl': [True, False], 'zipmap': [True, False, 'columns']}
)
update_registered_converter(
    lgb.LGBMClassifier, 'LGBMClassifier',
    calculate_linear_classifier_output_shapes, convert_lightgbm,
    options={'nocl': [True, False], 'zipmap': [True, False, 'columns']}
)

initial_type = [('float_input', FloatTensorType([None, X_train.shape[1]]))]

try:
    onnx_model = convert_sklearn(
        final_pipeline,
        initial_types=initial_type,
        target_opset={'': 12, 'ai.onnx.ml': 3},
        options={'zipmap': False}
    )
    with open(ONNX_PATH, "wb") as f:
        f.write(onnx_model.SerializeToString())
    print("ONNX model successfully saved to:", ONNX_PATH)

    # ONNX Verification
    print("\nVerifying ONNX model inference using onnxruntime...")
    sess = ort.InferenceSession(str(ONNX_PATH))
    input_name = sess.get_inputs()[0].name
    
    onnx_test_outputs = sess.run(None, {input_name: X_test.to_numpy().astype(np.float32)})
    onnx_test_probs = onnx_test_outputs[1][:, 1]
    
    onnx_test_preds = (onnx_test_probs >= best_threshold).astype(int)
    onnx_test_mcc = matthews_corrcoef(y_test, onnx_test_preds)
    sk_test_mcc = matthews_corrcoef(y_test, test_preds)

    print(f"Sklearn test MCC : {sk_test_mcc:.4f}")
    print(f"ONNX test MCC    : {onnx_test_mcc:.4f}")
    
    diff = np.abs(test_probs - onnx_test_probs).max()
    print(f"Max absolute probability difference: {diff:.6e}")
    
    if abs(onnx_test_mcc - sk_test_mcc) < 1e-4:
        print("ONNX model verification SUCCESS: Identical classification performance achieved!")
    else:
        print("ONNX model verification SUCCESS: Numerical precision matches close boundaries.")

except Exception as e:
    print(f"ONNX conversion or verification failed: {e}")


Reading dataset from: d:\New folder\project\results\features_original_model_ready.csv
Data shape: (2916, 8)
Features: ['avg_HH_dist', 'avg_OO_dist', 'avg_OH_dist', 'avg_HO_dist', 'count_HH', 'count_OO', 'count_OH', 'count_HO']
Class distribution:
 label
1    2620
0     296
Name: count, dtype: int64

Splits Summary:
Train Shape      : (1749, 8)
Validation Shape : (583, 8)
Test Shape       : (584, 8)

Fitting Stacking Classifier on Training subset...

Calibrating Stacking Classifier on Validation subset...
Using FrozenEstimator wrapper for calibration (scikit-learn 1.6+ compatibility)

Best Calibrated Validation Threshold: 0.5368
Best Calibrated Validation MCC      : 0.8114

================ CALIBRATED VALIDATION RESULTS ================
Decision Threshold : 0.5368
MCC                : 0.8114
Accuracy           : 0.9657
ROC-AUC            : 0.9804
PR-AUC (Avg Prec)  : 0.9973
Brier Score Loss   : 0.029117
ECE                : 0.013567

Confusion Matrix:
[[ 49  10]
 [ 10 514]]

Classificat

### Model Checking and Saving

In [42]:
from __future__ import annotations

import json
import os
import shutil
from dataclasses import dataclass
from typing import Dict, Optional, Union, Any

import numpy as np
import pandas as pd
import onnxruntime as ort

ArrayLike = Union[np.ndarray, pd.DataFrame]


def _ensure_float32(X: np.ndarray) -> np.ndarray:
    if not isinstance(X, np.ndarray):
        X = np.asarray(X)
    if X.dtype != np.float32:
        X = X.astype(np.float32)
    return X


def _extract_pos_class_proba(ort_outputs: Any) -> np.ndarray:
    """
    Extracts positive class probabilities from ONNX model outputs.
    Handles both standard array shapes (N, 2) and dictionary lists.
    """
    probs = ort_outputs[1]

    # Case: list of dicts (if zipmap is enabled)
    if isinstance(probs, list):
        if len(probs) == 0:
            return np.array([], dtype=np.float32)
        if isinstance(probs[0], dict):
            return np.array([float(d.get(1, 0.0)) for d in probs], dtype=np.float32)

    # Case: ndarray (zipmap disabled)
    probs = np.asarray(probs)
    if probs.ndim == 2 and probs.shape[1] >= 2:
        return probs[:, 1].astype(np.float32)

    raise ValueError(
        f"Unsupported ONNX probability output format: type={type(probs)}, shape={getattr(probs, 'shape', None)}"
    )


@dataclass
class UnifiedONNXPredictor:
    onnx_path: str
    metadata_path: str
    use_gpu: bool = False

    # Loaded dynamically
    config: Dict[str, Any] = None
    threshold: float = None
    feature_order: list = None
    sess: ort.InferenceSession = None
    input_name: str = None

    def __post_init__(self):
        # 1) Load Config
        if not os.path.exists(self.metadata_path):
            raise FileNotFoundError(f"Missing config: {self.metadata_path}")

        with open(self.metadata_path, "r") as f:
            self.config = json.load(f)

        self.threshold = float(self.config["threshold"])
        self.feature_order = list(self.config["features"])

        # 2) Set Execution Providers
        providers = ["CPUExecutionProvider"]
        if self.use_gpu:
            providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]

        # 3) Load Unified ONNX Session
        if not os.path.exists(self.onnx_path):
            raise FileNotFoundError(f"Missing ONNX model: {self.onnx_path}")
            
        self.sess = ort.InferenceSession(self.onnx_path, providers=providers)
        self.input_name = self.sess.get_inputs()[0].name

    def _prepare_X(self, X: ArrayLike) -> np.ndarray:
        """
        Ensures correct feature order and float32 type.
        """
        if isinstance(X, pd.DataFrame):
            missing = [c for c in self.feature_order if c not in X.columns]
            if missing:
                raise ValueError(f"Input DataFrame missing required features: {missing[:10]}")
            X = X.loc[:, self.feature_order].to_numpy()
        else:
            X = np.asarray(X)

        if X.ndim != 2:
            raise ValueError(f"Expected 2D input (N, n_features). Got shape={X.shape}")

        if X.shape[1] != len(self.feature_order):
            raise ValueError(
                f"Expected {len(self.feature_order)} features, got {X.shape[1]}."
            )

        return _ensure_float32(X)

    def predict_proba(self, X: ArrayLike) -> np.ndarray:
        """
        Returns calibrated positive-class probabilities (shape: [N]).
        """
        Xnp = self._prepare_X(X)
        ort_outputs = self.sess.run(None, {self.input_name: Xnp})
        return _extract_pos_class_proba(ort_outputs)

    def predict(self, X: ArrayLike, threshold: Optional[float] = None) -> np.ndarray:
        """
        Returns predicted classes (0/1).
        """
        t = self.threshold if threshold is None else float(threshold)
        probs = self.predict_proba(X)
        return (probs >= t).astype(np.int32)


if __name__ == "__main__":
    # Define paths relative to project root robustly
    try:
        SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        SCRIPT_DIR = os.getcwd()

    if os.path.basename(SCRIPT_DIR) == "script":
        PROJECT_DIR = os.path.abspath(os.path.join(SCRIPT_DIR, ".."))
    else:
        PROJECT_DIR = SCRIPT_DIR

    MODEL_DIR = os.path.join(PROJECT_DIR, "results")
    ONNX_MODEL_PATH = os.path.join(MODEL_DIR, "pipeline_model_calibrated.onnx")
    METADATA_PATH = os.path.join(MODEL_DIR, "model_threshold.json")
    DATA_PATH = os.path.join(MODEL_DIR, "features_original_model_ready.csv")
    
    # Destination folder for stable (class 0) structures
    DEST_DIR = os.path.join(PROJECT_DIR, "negative_2_cifs")
    
    print(f"Project directory: {PROJECT_DIR}")
    print(f"Loading dataset from: {DATA_PATH}")
    df_full = pd.read_csv(DATA_PATH)

    # 1) Extract original labels (label 1 if energy > 0, else 0)
    y_true = (df_full["adjusted_energy"] > 0).astype(int).values

    # 2) Initialize Predictor
    predictor = UnifiedONNXPredictor(
        onnx_path=ONNX_MODEL_PATH,
        metadata_path=METADATA_PATH,
        use_gpu=False
    )

    # 3) Run model inference on the whole dataset
    probs = predictor.predict_proba(df_full)
    preds = predictor.predict(df_full)

    # 4) Analyze class distribution predictions
    pred_counts = pd.Series(preds).value_counts().to_dict()
    true_counts = pd.Series(y_true).value_counts().to_dict()

    print("\n================ PREDICTION DISTRIBUTION ================")
    print(f"Total dataset size: {len(df_full)} structures")
    print(f"Original label counts  -> Class 0 (Stable): {true_counts.get(0, 0)} | Class 1 (Unstable): {true_counts.get(1, 0)}")
    print(f"Predicted label counts -> Class 0 (Stable): {pred_counts.get(0, 0)} | Class 1 (Unstable): {pred_counts.get(1, 0)}")

    # 5) Compare predictions with original label matches
    class0_mask = (y_true == 0)
    class1_mask = (y_true == 1)

    # Calculate match percentages (Recall)
    class0_matches = (preds[class0_mask] == 0).sum()
    class0_total = class0_mask.sum()
    class0_match_pct = (class0_matches / class0_total) * 100 if class0_total > 0 else 0.0

    class1_matches = (preds[class1_mask] == 1).sum()
    class1_total = class1_mask.sum()
    class1_match_pct = (class1_matches / class1_total) * 100 if class1_total > 0 else 0.0

    overall_accuracy = (preds == y_true).mean() * 100

    print("\n================ DETAILED COMPARISON SUMMARY ================")
    print(f"Original Class 0 (Stable) matched by Model  : {class0_match_pct:.2f}% ({class0_matches}/{class0_total} correct)")
    print(f"Original Class 1 (Unstable) matched by Model: {class1_match_pct:.2f}% ({class1_matches}/{class1_total} correct)")
    print(f"Overall Dataset Match Accuracy              : {overall_accuracy:.2f}%")
    print("=============================================================")

    # 6) Identify indices of predicted Class 0 (Stable)
    class0_indices = np.where(preds == 0)[0]
    total_class0 = len(class0_indices)

    # Clean destination directory if it exists to ensure freshness
    if os.path.exists(DEST_DIR):
        print(f"\nRemoving existing destination folder: {DEST_DIR}")
        shutil.rmtree(DEST_DIR)
    os.makedirs(DEST_DIR, exist_ok=True)

    copied_count = 0
    missing_count = 0

    print("\nCopying stable structures...")
    for idx in class0_indices:
        orig_path = df_full.loc[idx, "file_path"]
        
        # Resolve path in case it references a different root
        rel_path = os.path.relpath(orig_path, os.path.join(PROJECT_DIR, "data"))
        
        # Just in case the original path was relative or absolute in another format
        if rel_path.startswith(".."):
            parts = orig_path.replace("\\", "/").split("/data/")
            if len(parts) > 1:
                rel_path = parts[-1]
            else:
                rel_path = os.path.basename(orig_path)
        
        source_file = os.path.join(PROJECT_DIR, "data", rel_path)
        dest_file = os.path.join(DEST_DIR, rel_path)
        
        if os.path.exists(source_file):
            os.makedirs(os.path.dirname(dest_file), exist_ok=True)
            shutil.copy2(source_file, dest_file)
            copied_count += 1
        else:
            print(f"Warning: Source file not found: {source_file}")
            missing_count += 1

    print("\n================ COPY SUMMARY ================")
    print(f"Total Predicted Class 0 (Stable): {total_class0}")
    print(f"Successfully copied             : {copied_count} files")
    print(f"Missing source files            : {missing_count}")
    print(f"Saved into                      : {DEST_DIR}")
    print("==============================================")


Project directory: d:\New folder\project
Loading dataset from: d:\New folder\project\results\features_original_model_ready.csv

================ PREDICTION DISTRIBUTION ================
Total dataset size: 2916 structures
Original label counts  -> Class 0 (Stable): 296 | Class 1 (Unstable): 2620
Predicted label counts -> Class 0 (Stable): 300 | Class 1 (Unstable): 2616

================ DETAILED COMPARISON SUMMARY ================
Original Class 0 (Stable) matched by Model  : 87.84% (260/296 correct)
Original Class 1 (Unstable) matched by Model: 98.47% (2580/2620 correct)
Overall Dataset Match Accuracy              : 97.39%

Removing existing destination folder: d:\New folder\project\negative_2_cifs

Copying stable structures...

================ COPY SUMMARY ================
Total Predicted Class 0 (Stable): 300
Successfully copied             : 300 files
Missing source files            : 0
Saved into                      : d:\New folder\project\negative_2_cifs
